# SR-AttELM-H: reproduction of the experiments

The notebook clones the repository, downloads the Beijing Multi-Site Air Quality data set (UCI, https://doi.org/10.24432/C5RK5G),
checks the evaluation protocol against the frozen values of the article and runs:

1. the technique, its component-exclusion variants and LOCF under geometric and network-derived delays (Tables 12, 13);
2. the statistics and autocorrelation of the network-derived delays (Tables 10, 11, Fig. 8);
3. optionally, the official AGG core under the same protocol (several hours on a CPU runtime).

Use *Runtime → Run all*.

In [ ]:
# Configuration
REPO_URL = "https://github.com/ALAXAY/sr-attelm-h.git"
REPO_DIR = "sr-attelm-h"
RUN_AGG = False            # True: train and evaluate the official AGG core (several hours on a CPU runtime)
REGENERATE_DELAYS = False  # True: regenerate the network-derived delays instead of using the shipped realizations
THREADS = "8"              # CPU threads for PyTorch

In [ ]:
import os, subprocess, sys

def run(cmd, cwd="."):
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError("command failed: " + " ".join(cmd))

if not os.path.isdir(REPO_DIR):
    run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
os.chdir(REPO_DIR)
os.environ["SRATTELM_THREADS"] = THREADS
os.environ["AGG_THREADS"] = THREADS
print(os.getcwd())

In [ ]:
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

In [ ]:
# Data set (about 8 MB); the loader verifies the SHA-256 of the loaded array
import pathlib, urllib.request
UCI_URL = "https://archive.ics.uci.edu/static/public/501/beijing+multi+site+air+quality+data.zip"
if not list(pathlib.Path("data").glob("*.zip")):
    urllib.request.urlretrieve(UCI_URL, "data/beijing_multi_site_air_quality.zip")
sys.path.insert(0, os.path.abspath("srattelm"))
import protocol as PR
X, STATIONS = PR.load_raw()
print(X.shape, STATIONS)

In [ ]:
# Protocol check: delayed components and LOCF RMSE must coincide with the frozen protocol (Table 6)
import numpy as np
import run_experiments as RE
EXPECTED = {"A": ([31722, 31718, 31728], [0.5089, 0.5015, 0.5166]),
            "B": ([42042, 42044, 42038], [0.5714, 0.5749, 0.5554])}
for f, (n_exp, r_exp) in EXPECTED.items():
    _, tev = RE.views_geo(f)
    rows = RE.locf_rows(tev, f, "geometric")
    n = [int(r["n"]) for r in rows]
    rm = [r["RMSE"] for r in rows]
    assert n == n_exp and np.allclose(rm, r_exp, atol=1.5e-4), (f, n, rm)
    print("split", f, n, np.round(rm, 4), "- coincides with the frozen protocol")

In [ ]:
# Network-derived delays: optional regeneration, statistics (Tables 10, 11) and Fig. 8
if REGENERATE_DELAYS:
    run([sys.executable, "generate_delays.py"], cwd="netsim")
    run([sys.executable, "generate_training_delays.py"], cwd="netsim")
run([sys.executable, "delay_statistics.py"], cwd="netsim")
run([sys.executable, "make_figure.py"], cwd="netsim")
from IPython.display import Image, display
display(Image("netsim/out/Fig8_network_delays.png", width=900))

In [ ]:
# Technique, variants and LOCF under geometric and network-derived delays; component exclusion (Table 13)
run([sys.executable, "run_experiments.py"], cwd="srattelm")
run([sys.executable, "run_component_check.py"], cwd="srattelm")

In [ ]:
# Official AGG core (optional)
if RUN_AGG:
    run([sys.executable, "run_agg.py"], cwd="srattelm/agg")
else:
    print("RUN_AGG = False: the shipped srattelm/results_official_agg.csv is used")

In [ ]:
# Tables 12 and 13
import pandas as pd
from IPython.display import display
run([sys.executable, "summarize.py"], cwd="srattelm")
run([sys.executable, "summarize_components.py"], cwd="srattelm")
pd.set_option("display.width", 200)
Z = pd.read_csv("srattelm/summary_network.csv")
A = pd.read_csv("srattelm/availability_check_summary.csv")
display(Z)
display(A)

In [ ]:
# Comparison with the values reported in the article (Section 7.8)
ARTICLE_12 = {("LOCF", "network"): 0.7968, ("Official AGG core", "network"): 0.4974,
              ("Proposed technique", "geometric"): 0.2941, ("Proposed technique", "network"): 0.3894,
              ("Proposed technique, trained on geometric delays", "network"): 0.4214}
ARTICLE_13 = {("geometric", "spatial_pure"): 0.4088, ("geometric", "st_explicit"): 0.3749,
              ("geometric", "weights_av_explicit"): 0.2931, ("geometric", "weights_noav"): 0.3278,
              ("network", "spatial_pure"): 0.5568, ("network", "st_explicit"): 0.4873,
              ("network", "weights_av_explicit"): 0.3904, ("network", "weights_noav"): 0.4608}
for (lab, dl), v in ARTICLE_12.items():
    x = float(Z[(Z.label == lab) & (Z.test == dl)].RMSE.iloc[0])
    print(f"{lab:50s} {dl:9s} article {v:.4f}   this run {x:.4f}")
for (dl, var), v in ARTICLE_13.items():
    x = float(A[(A.delays == dl) & (A.variant == var)].RMSE_pooled.iloc[0])
    print(f"{var:50s} {dl:9s} article {v:.4f}   this run {x:.4f}")